# Houston delivery distance and carbon
Workbook Deliveries -> Azure Maps geocoding/road routing -> Delta -> Power BI.
Run the numbered workflow cells in order. Each step prints its actual data as indented JSON, including all rows and error details, rather than only status counts. Input key-named columns are excluded from printed output. API request and response examples appear beside the helper implementations that issue those requests.


## Define the enrichment helpers

The helper cells below define small, explicit operations. The runnable workflow later produces `validated`, `geocoded`, `routed`, and `enriched` snapshots in separate cells. Defining helpers does not make API calls.

For each delivery, the helpers validate the input, geocode the supplier and plant addresses, request a road route, and calculate carbon estimates. Address and directional route caches avoid repeating successful requests within one refresh. Coordinates use `[longitude, latitude]` throughout.


### Shared HTTP client

`MapsHttpClient` handles authentication, pacing, retries, and response validation for both Maps endpoints. Requests are sequential and paced, with up to four attempts for network failures and retryable HTTP errors. Raw upstream error bodies and credentials are not copied into results.

Microsoft documentation: [Azure Maps authentication](https://learn.microsoft.com/azure/azure-maps/azure-maps-authentication) and [authentication best practices](https://learn.microsoft.com/azure/azure-maps/authentication-best-practices).


In [ ]:
import asyncio
import csv
import io
import json
import logging
import math
from collections import Counter
from collections.abc import Awaitable, Callable
from datetime import UTC, datetime
from typing import Any

import httpx
import notebookutils
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType

logger = logging.getLogger(__name__)
Row = dict[str, Any]
TokenProvider = Callable[[], Awaitable[str]]
API_VERSION = "2025-01-01"
ADDRESS_FIELDS = ("Street", "City", "RegionCode", "PstlCode", "CtryCode")
RESULT_FIELDS = {
    "Driving_distance_km",
    "CO2_emitted_kg",
    "Carbon_cost_USD",
    "Carbon_price_USD_per_tonne",
    "Route_geojson",
    "Route_Status",
    "Route_Error_Reason",
    "Route_Error_Detail",
    "Refreshed_UTC",
    "Supplier_address",
    "Plant_address",
    "Supplier_matched_address",
    "Plant_matched_address",
    "Supplier_route_latitude",
    "Supplier_route_longitude",
    "Plant_route_latitude",
    "Plant_route_longitude",
    "Input_signature",
}


def print_stage(stage: str, rows: list[Row]) -> None:
    safe_rows = [
        {key: value for key, value in row.items() if "key" not in key.lower()}
        for row in rows
    ]
    print(json.dumps({
        "stage": stage,
        "row_count": len(rows),
        "statuses": dict(Counter(row["Route_Status"] for row in rows if "Route_Status" in row)),
        "rows": safe_rows,
    }, indent=2))


def finite_number(value: Any, label: str, minimum: float = 0) -> float:
    if isinstance(value, bool):
        raise ValueError(f"{label} must be a finite number >= {minimum}.")
    result = float(value)
    if not math.isfinite(result) or result < minimum:
        raise ValueError(f"{label} must be a finite number >= {minimum}.")
    return result


def delivery_address(row: Row, prefix: str) -> str:
    values = [str(row.get(f"{prefix}_{name}") or "").strip() for name in ADDRESS_FIELDS]
    if not all(values):
        raise ValueError(f"{prefix} address is incomplete.")
    return ", ".join(values)


def coordinate(value: Any) -> list[float]:
    if not isinstance(value, list) or len(value) < 2:
        raise ValueError("Azure Maps returned an invalid coordinate.")
    lon, lat = float(value[0]), float(value[1])
    if not math.isfinite(lon) or not math.isfinite(lat) or abs(lon) > 180 or abs(lat) > 90:
        raise ValueError("Azure Maps returned an out-of-range coordinate.")
    return [lon, lat]


class MapsHttpClient:
    """Only handles authentication, pacing, retries, and HTTP response validation."""

    def __init__(
        self,
        http: httpx.AsyncClient,
        token_provider: TokenProvider | None,
        client_id: str,
        *,
        pace_seconds: float = 0.25,
        subscription_key_provider: TokenProvider | None = None,
    ) -> None:
        self.http = http
        self.token_provider = token_provider
        self.client_id = client_id
        self.pace_seconds = pace_seconds
        if (token_provider is None) == (subscription_key_provider is None):
            raise ValueError("Supply exactly one Maps credential provider.")
        self.subscription_key_provider = subscription_key_provider

    async def request(self, path: str, body: Row | None = None, **params: str) -> Row:
        # Both Maps endpoints use api-version=2025-01-01. No body means GET;
        # a GeoJSON body means POST. Credentials stay in headers, never URLs.
        # This notebook uses subscription-key: <key read from Key Vault>.
        # The alternative is Authorization: Bearer <token> plus x-ms-client-id.
        for attempt in range(4):
            if self.subscription_key_provider is not None:
                headers = {"subscription-key": await self.subscription_key_provider()}
            else:
                assert self.token_provider is not None
                headers = {
                    "Authorization": f"Bearer {await self.token_provider()}",
                    "x-ms-client-id": self.client_id,
                }
            headers["Content-Type"] = "application/geo+json"
            # Pace every attempt; retry network failures and transient HTTP errors
            # up to four attempts total, honoring numeric Retry-After (max 60s).
            await asyncio.sleep(self.pace_seconds)
            try:
                response = await self.http.request(
                    "GET" if body is None else "POST",
                    f"https://atlas.microsoft.com/{path}",
                    params={"api-version": API_VERSION, **params},
                    json=body,
                    headers=headers,
                )
            except httpx.RequestError:
                if attempt < 3:
                    await asyncio.sleep(2**attempt)
                    continue
                raise RuntimeError("MAPS_NETWORK_ERROR") from None
            if response.status_code in {408, 429, 500, 502, 503, 504} and attempt < 3:
                retry = response.headers.get("Retry-After", "")
                await asyncio.sleep(min(float(retry), 60) if retry.isdigit() else 2**attempt)
                continue
            if response.status_code != 200:
                # Do not expose credential-bearing request objects or arbitrary upstream bodies.
                raise RuntimeError(f"MAPS_HTTP_{response.status_code}")
            try:
                payload = response.json()
            except ValueError:
                raise RuntimeError("MAPS_INVALID_JSON") from None
            if not isinstance(payload, dict):
                raise RuntimeError("MAPS_INVALID_RESPONSE")
            return payload
        raise RuntimeError("MAPS_RETRY_EXHAUSTED")


## Validation helpers

`validate_deliveries` creates new rows, removes key-named columns, and clears all prior refresh results. Valid rows receive `VALIDATED`; invalid rows receive `ERROR`. No network calls occur here.

Each later stage copies its input rows, preserving earlier snapshots for inspection. `record_error` clears derived results and records safe diagnostics so errors cannot carry stale geometry or carbon estimates into the saved output. Invalid delivery data or a failed Maps call produces an `ERROR` row while other deliveries continue.


In [ ]:
def record_error(row: Row, error: ValueError | TypeError | RuntimeError) -> None:
    keep = {"Supplier_address", "Plant_address", "Carbon_price_USD_per_tonne", "Refreshed_UTC"}
    row.update(dict.fromkeys(RESULT_FIELDS - keep))
    row.update(
        Route_Status="ERROR",
        Route_Error_Reason=str(error) if isinstance(error, RuntimeError) else "INVALID_DATA",
        Route_Error_Detail="Routing failed; review input or API access."
        if isinstance(error, RuntimeError) else str(error),
    )


def require_stage(rows: list[Row], expected: str) -> None:
    if any(row.get("Route_Status") not in {expected, "ERROR"} for row in rows):
        raise ValueError(f"Expected {expected} rows; run the preceding stage first.")


def validate_deliveries(
    rows: list[Row], carbon_price: float = 100, max_rows: int = 1000
) -> list[Row]:
    price = finite_number(carbon_price, "USD per tonne")
    if not 1 <= len(rows) <= max_rows:
        raise ValueError(f"Expected 1..{max_rows} deliveries.")
    ids = Counter(str(row.get("Delivery_ID") or "").strip() for row in rows)
    results = []
    for row in rows:
        out = {
            key: value for key, value in row.items()
            if key not in RESULT_FIELDS and "key" not in key.lower()
        }
        out.update(dict.fromkeys(RESULT_FIELDS))
        out.update(
            Carbon_price_USD_per_tonne=price,
            Refreshed_UTC=datetime.now(UTC).isoformat(),
        )
        try:
            delivery_id = str(row.get("Delivery_ID") or "").strip()
            if not delivery_id or ids[delivery_id] != 1:
                raise ValueError("Delivery_ID must be nonempty and unique.")
            finite_number(row.get("Emissions_factor_kgCO2_per_km"), "CO2 factor")
            if finite_number(row.get("Trip_multiplier"), "Trip multiplier") == 0:
                raise ValueError("Trip multiplier must be positive.")
            origin = delivery_address(row, "ManfactSupplier")
            destination = delivery_address(row, "Plant")
            out.update(Supplier_address=origin, Plant_address=destination, Route_Status="VALIDATED")
        except (ValueError, TypeError) as error:
            record_error(out, error)
        results.append(out)
    return results


## Geocoding helpers

`geocode_address` issues the `client.request("geocode", ...)` call in the next cell. `geocode_deliveries` applies it to validated rows, caching successful address matches within that stage run. It stores matched addresses and road-access coordinates, then marks each successful row `GEOCODED`. It does not request routes or calculate carbon.

### Geocoding REST exchange

Each uncached full address is converted to coordinates. `httpx` URL-encodes the query parameters. The request has no JSON body; the subscription key comes from Key Vault and is sent only in a header.

```http
GET /geocode?api-version=2025-01-01&query=1500%20McKinney%20St%2C%20Houston%2C%20TX%2C%2077010%2C%20US&top=1 HTTP/1.1
Host: atlas.microsoft.com
subscription-key: <key retrieved at runtime>
Content-Type: application/geo+json
```

Example HTTP 200 response, limited to relevant fields. Coordinates and matched addresses are illustrative, not verified API results.

```json
{
  "type": "FeatureCollection",
  "features": [{
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [-95.36, 29.75]},
    "properties": {
      "confidence": "High",
      "matchCodes": ["Good"],
      "address": {"formattedAddress": "1500 McKinney St, Houston, TX 77010"},
      "geocodePoints": [{
        "usageTypes": ["Route"],
        "geometry": {"type": "Point", "coordinates": [-95.361, 29.751]}
      }]
    }
  }]
}
```

The helper accepts only high-confidence `Good` matches without `Ambiguous` or `UpHierarchy` codes. It prefers a point tagged `Route`, falling back to the feature point when no route point exists. The plant address goes through the same process. Coordinates use `[longitude, latitude]`.

Microsoft documentation: [Get Geocoding REST API (2025-01-01)](https://learn.microsoft.com/rest/api/maps/search/get-geocoding?view=rest-maps-2025-01-01&preserve-view=true) and [forward-geocoding best practices](https://learn.microsoft.com/azure/azure-maps/how-to-use-best-practices-for-search#best-practices-for-forward-geocoding).


In [ ]:
async def geocode_address(client: MapsHttpClient, address: str) -> tuple[list[float], str]:
    # GET /geocode: the full address is a query parameter, not a JSON body.
    data = await client.request("geocode", query=address, top="1")
    try:
        feature = data["features"][0]
        props = feature["properties"]
        codes = props.get("matchCodes", [])
        if (
            props.get("confidence") != "High" or "Good" not in codes
            or any(code in codes for code in ("Ambiguous", "UpHierarchy"))
        ):
            raise ValueError("Geocoded address needs review.")
        points = [
            point for point in props.get("geocodePoints", [])
            if "Route" in point.get("usageTypes", [])
        ]
        # Prefer road-access coordinates over the address's feature point.
        point = points[0] if points else feature
        coords = coordinate(point["geometry"]["coordinates"])
        matched = props["address"]["formattedAddress"]
        if not isinstance(matched, str) or not matched.strip():
            raise ValueError("Missing matched address.")
    except (KeyError, IndexError, TypeError):
        raise ValueError("Azure Maps returned an incomplete geocode.") from None
    return coords, matched


async def geocode_deliveries(rows: list[Row], client: MapsHttpClient) -> list[Row]:
    require_stage(rows, "VALIDATED")
    cache: dict[str, tuple[list[float], str]] = {}
    results = []
    for row in rows:
        out = dict(row)
        if row["Route_Status"] != "ERROR":
            try:
                for address_field, prefix in (("Supplier_address", "Supplier"), ("Plant_address", "Plant")):
                    address = row[address_field]
                    if address not in cache:
                        cache[address] = await geocode_address(client, address)
                    (lon, lat), matched = cache[address]
                    out.update({
                        f"{prefix}_matched_address": matched,
                        f"{prefix}_route_longitude": lon,
                        f"{prefix}_route_latitude": lat,
                    })
                out["Route_Status"] = "GEOCODED"
            except (ValueError, TypeError, RuntimeError) as error:
                record_error(out, error)
        results.append(out)
    return results


## Road-routing helpers

`request_road_route` issues the `client.request("route/directions", body=...)` call in the next cell and validates the returned road path. `route_deliveries` caches successful supplier-to-plant routes within this stage run and marks successful rows `ROUTED`. It never geocodes addresses and does not calculate carbon. Reversing the origin and destination creates a different cache entry.

### Routing REST exchange

The endpoint takes two geocoded waypoints and returns a driving route optimized without traffic. Waypoint 0 is the supplier; waypoint 1 is the plant. The reverse trip is a different cache entry. Driving mode is not restricted commercial-truck routing.

```http
POST /route/directions?api-version=2025-01-01 HTTP/1.1
Host: atlas.microsoft.com
subscription-key: <key retrieved at runtime>
Content-Type: application/geo+json
```

JSON request body:

```json
{
  "type": "FeatureCollection",
  "features": [
    {"type": "Feature",
     "geometry": {"type": "Point", "coordinates": [-95.361, 29.751]},
     "properties": {"pointIndex": 0, "pointType": "waypoint"}},
    {"type": "Feature",
     "geometry": {"type": "Point", "coordinates": [-95.35, 29.76]},
     "properties": {"pointIndex": 1, "pointType": "waypoint"}}
  ],
  "travelMode": "driving",
  "optimizeRoute": "fastestWithoutTraffic",
  "maxRouteCount": 1,
  "routeOutputOptions": ["routePath"]
}
```

Example HTTP 200 response, with other fields and road vertices omitted. Distance and geometry are illustrative, not an actual route result.

```json
{
  "type": "FeatureCollection",
  "features": [{
    "type": "Feature",
    "properties": {"type": "RoutePath", "distanceInMeters": 12345.6},
    "geometry": {
      "type": "MultiLineString",
      "coordinates": [[[-95.361, 29.751], [-95.35, 29.76]]]
    }
  }]
}
```

The helper requires exactly one `RoutePath`, converts its distance from meters to kilometers, and normalizes `LineString` or `MultiLineString` geometry into a `MultiLineString`. It does not substitute straight-line distance when routing fails. Calls run sequentially on the driver, never Spark executors.

Microsoft documentation: [Post Route Directions REST API (2025-01-01)](https://learn.microsoft.com/rest/api/maps/route/post-route-directions?view=rest-maps-2025-01-01&preserve-view=true) for waypoint GeoJSON, routing options, distance, and route-path responses.


In [ ]:
async def request_road_route(
    client: MapsHttpClient, start: list[float], end: list[float]
) -> Row:
    # POST /route/directions: GeoJSON waypoints are [longitude, latitude].
    # Supplier is pointIndex 0; plant is pointIndex 1.
    data = await client.request(
        "route/directions",
        body={
            "type": "FeatureCollection",
            "features": [
                {
                    "type": "Feature",
                    "geometry": {"type": "Point", "coordinates": point},
                    "properties": {"pointIndex": index, "pointType": "waypoint"},
                }
                for index, point in enumerate((start, end))
            ],
            "travelMode": "driving",
            "optimizeRoute": "fastestWithoutTraffic",
            "maxRouteCount": 1,
            "routeOutputOptions": ["routePath"],
        },
    )
    try:
        paths = [
            feature for feature in data["features"]
            if feature.get("properties", {}).get("type") == "RoutePath"
        ]
        if len(paths) != 1:
            raise ValueError("Expected exactly one Azure Maps RoutePath.")
        path = paths[0]
        distance = finite_number(path["properties"]["distanceInMeters"], "Road meters")
        geometry = path["geometry"]
        if geometry["type"] == "LineString":
            lines = [geometry["coordinates"]]
        elif geometry["type"] == "MultiLineString":
            lines = geometry["coordinates"]
        else:
            raise ValueError("Expected road LineString or MultiLineString.")
        if not lines or any(len(line) < 2 for line in lines):
            raise ValueError("Azure Maps returned an empty road path.")
        lines = [[coordinate(point) for point in line] for line in lines]
    except (KeyError, IndexError, TypeError):
        raise ValueError("Azure Maps returned an incomplete road path.") from None
    # Azure Maps returns road meters; downstream carbon calculations use km.
    return {
        "Driving_distance_km": distance / 1000,
        "Route_geojson": json.dumps({"type": "MultiLineString", "coordinates": lines}, allow_nan=False),
    }


async def route_deliveries(rows: list[Row], client: MapsHttpClient) -> list[Row]:
    require_stage(rows, "GEOCODED")
    cache: dict[tuple[str, str], Row] = {}
    results = []
    for row in rows:
        out = dict(row)
        if row["Route_Status"] != "ERROR":
            try:
                pair = row["Supplier_address"], row["Plant_address"]
                if pair not in cache:
                    start = [row["Supplier_route_longitude"], row["Supplier_route_latitude"]]
                    end = [row["Plant_route_longitude"], row["Plant_route_latitude"]]
                    cache[pair] = await request_road_route(client, start, end)
                out.update(cache[pair])
                out["Route_Status"] = "ROUTED"
            except (ValueError, TypeError, RuntimeError) as error:
                record_error(out, error)
        results.append(out)
    return results


## Carbon and map-point helpers

`calculate_carbon` uses routed distance, the original emissions factor, trip multiplier, and validated carbon price. It makes no network calls. Only rows with completed calculations receive the final `OK` status. `route_points` later expands these successful routes for the map.


In [ ]:
def calculate_carbon(rows: list[Row]) -> list[Row]:
    require_stage(rows, "ROUTED")
    results = []
    for row in rows:
        out = dict(row)
        if row["Route_Status"] != "ERROR":
            try:
                factor = finite_number(row["Emissions_factor_kgCO2_per_km"], "CO2 factor")
                multiplier = finite_number(row["Trip_multiplier"], "Trip multiplier")
                if multiplier == 0:
                    raise ValueError("Trip multiplier must be positive.")
                price = finite_number(row["Carbon_price_USD_per_tonne"], "USD per tonne")
                co2 = finite_number(row["Driving_distance_km"] * factor * multiplier, "Estimated CO2")
                cost = finite_number(co2 / 1000 * price, "Carbon cost")
                out.update(
                    Emissions_factor_kgCO2_per_km=factor,
                    Trip_multiplier=multiplier,
                    CO2_emitted_kg=co2,
                    Carbon_cost_USD=cost,
                    Route_Status="OK",
                )
            except (ValueError, TypeError) as error:
                record_error(out, error)
        results.append(out)
    logger.info(
        "delivery_enrichment_completed",
        extra={"rows": len(results), "successful": sum(r["Route_Status"] == "OK" for r in results)},
    )
    return results


def route_points(rows: list[Row]) -> list[Row]:
    # Step 5: Expand successful road paths into ordered Power BI map vertices.
    # Path_ID separates segments; Point_sequence preserves order within a segment.
    points = []
    for row in rows:
        if row["Route_Status"] != "OK":
            continue
        geometry = json.loads(row["Route_geojson"])
        for segment, line in enumerate(geometry["coordinates"]):
            for sequence, (lon, lat) in enumerate(line):
                points.append(
                    {
                        "Delivery_ID": row["Delivery_ID"],
                        "Path_ID": f"{row['Delivery_ID']}:{segment}",
                        "Route_segment": segment,
                        "Point_sequence": sequence,
                        "Longitude": lon,
                        "Latitude": lat,
                    }
                )
    return points


## Configure the refresh

Set the input CSV path, carbon price in USD per metric tonne, and maximum delivery count. The CSV path is relative to the attached Lakehouse. Keep the Key Vault URL and secret name here, but never paste the secret value into the notebook.

This run uses subscription-key authentication. The Maps key is read from Key Vault at runtime; it is never stored in notebook source or data. `MAPS_CLIENT_ID` is only needed by the helpers' alternative bearer-token authentication path.

Each CSV row needs a nonempty, unique `Delivery_ID`, a nonnegative `Emissions_factor_kgCO2_per_km`, and a positive `Trip_multiplier`. Both address prefixes, `ManfactSupplier` and `Plant`, require `_Street`, `_City`, `_RegionCode`, `_PstlCode`, and `_CtryCode` columns. Preserve the existing `ManfactSupplier` spelling.

Microsoft documentation: [Shared Key authentication with Azure Key Vault](https://learn.microsoft.com/azure/azure-maps/how-to-secure-daemon-app#scenario-shared-key-authentication-with-azure-key-vault) and [Fabric credential utilities and Key Vault permissions](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials#constraints-and-safety).


In [ ]:
MAPS_CLIENT_ID = 'cc860026-0f34-4d46-85e1-41552df6abfb'
CARBON_PRICE_USD_PER_TONNE = 100.0
KEY_VAULT_URL = 'https://kv-scroute-fabric-rjb.vault.azure.net/'
MAPS_SECRET_NAME = 'azure-maps-key'
INPUT_FILE = 'Files/sample/workbook_deliveries.csv'
MAX_ROWS = 1000


## Step 1: Read and validate deliveries

Run in Fabric with the delivery Lakehouse attached. The next cell rejects files larger than 4 MiB and parses the CSV into driver-side dictionaries. CSV values initially arrive as strings. This stage does not access Key Vault or Azure Maps.

`validate_deliveries` checks the batch size, unique IDs, full addresses, emissions factors, and trip multipliers. Inspect `validated` before making billable requests. Invalid rows are retained as `ERROR` rows and skipped by subsequent stages.

Run the numbered cells in order. Each stage prints all result rows, row counts, and available status counts as indented JSON, and leaves a separate inspectable result. Step 1 prints both the input (without key-named columns) and validated rows. Route geometry and route points are printed in full, so later output can be large. After changing upstream inputs, rerun all downstream stages. Rerunning a network stage starts fresh stage-local caches and can make billable requests again.

Microsoft documentation: [NotebookUtils file-system utilities](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-file-system) for default Lakehouse paths, file properties, and `fs.head` reads.


In [ ]:

spark = SparkSession.builder.getOrCreate()

input_directory, input_name = INPUT_FILE.rsplit("/", 1)
file_info = next(
    entry for entry in notebookutils.fs.ls(input_directory) if entry.name == input_name
)
if file_info.size > 4 * 1024 * 1024:
    raise ValueError("Input exceeds the 4 MiB driver-side file limit; split the batch.")

input_text = notebookutils.fs.head(INPUT_FILE, 4 * 1024 * 1024)
deliveries = list(csv.DictReader(io.StringIO(input_text)))
print_stage("input", deliveries)
validated = validate_deliveries(deliveries, CARBON_PRICE_USD_PER_TONNE, MAX_ROWS)
print_stage("validated", validated)


## Step 2: Geocode the addresses

Retrieve the Maps key through Fabric's credential helper, then resolve validated supplier and plant addresses. Calls run sequentially on the driver with a 60-second HTTP timeout. The HTTP client is closed at the end of the cell.

Inspect `geocoded` for matched addresses, road-access longitude/latitude, and `GEOCODED` or `ERROR` status. `validated` remains unchanged. This stage makes only geocoding requests, not routing requests.

The request and response examples are next to `geocode_address` in the Geocoding helpers section, where the endpoint is called.

Microsoft documentation: [Retrieve a Key Vault secret with `notebookutils.credentials.getSecret`](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials#get-secret) and [Get Geocoding REST API (2025-01-01)](https://learn.microsoft.com/rest/api/maps/search/get-geocoding?view=rest-maps-2025-01-01&preserve-view=true).


In [ ]:
# Fetch the key at runtime; do not print it or include it in result rows.
maps_key = notebookutils.credentials.getSecret(KEY_VAULT_URL, MAPS_SECRET_NAME)

async def maps_key_provider() -> str:
    return maps_key

async with httpx.AsyncClient(timeout=60) as http:
    client = MapsHttpClient(
        http, None, MAPS_CLIENT_ID, subscription_key_provider=maps_key_provider
    )
    # GET /geocode for each uncached supplier/plant address, on the driver only.
    geocoded = await geocode_deliveries(validated, client)
print_stage("geocoded", geocoded)


## Step 3: Request road routes

Use the coordinates from `geocoded` to request driving routes. No geocoding occurs in this stage. Inspect `routed` for `Driving_distance_km`, `Route_geojson`, and `ROUTED` or `ERROR` status. Carbon fields remain null until the next step.

Earlier snapshots remain unchanged. Successful directional routes are reused for deliveries sharing the same supplier and plant addresses during this stage run.

The request and response examples are next to `request_road_route` in the Road-routing helpers section, where the endpoint is called.

Microsoft documentation: [Post Route Directions REST API (2025-01-01)](https://learn.microsoft.com/rest/api/maps/route/post-route-directions?view=rest-maps-2025-01-01&preserve-view=true).


In [ ]:
async with httpx.AsyncClient(timeout=60) as http:
    client = MapsHttpClient(
        http, None, MAPS_CLIENT_ID, subscription_key_provider=maps_key_provider
    )
    # POST /route/directions for each uncached directional pair; reuse the Maps key.
    routed = await route_deliveries(geocoded, client)
print_stage("routed", routed)


## Step 4: Calculate carbon estimates

Calculate CO2 and USD carbon cost from `routed` using the formulas below. This step makes no REST calls. Inspect `enriched` for the final results: successful rows are `OK`, and failed rows retain safe error details with null route metrics.

The `validated`, `geocoded`, and `routed` snapshots remain available to compare intermediate results. No tables or exports have been overwritten yet.

One row is one vehicle trip. CO2 is an illustrative estimate calculated locally, not an Azure Maps output:

```text
Driving_distance_km = distanceInMeters / 1000
CO2_emitted_kg = Driving_distance_km * Emissions_factor_kgCO2_per_km * Trip_multiplier
Carbon_cost_USD = CO2_emitted_kg / 1000 * Carbon_price_USD_per_tonne
```

For example, 12.3456 km at 0.27 kg CO2/vehicle-km with a multiplier of 2 produces 6.666624 kg CO2. At 100 USD/tonne, the carbon cost is 0.6666624 USD. Part quantity does not multiply this vehicle-trip estimate.


In [ ]:
# Calculate vehicle-trip CO2 and carbon cost locally, not through an API.
enriched = calculate_carbon(routed)
print_stage("enriched", enriched)


## Step 5: Prepare typed delivery data

Build an explicit Spark schema so numeric columns remain numeric even when some or all route results are missing. Convert numeric strings to doubles and represent empty, invalid, or non-finite numeric values as null. Other columns are stored as strings.

This step prepares `typed` and `schema` for Delta storage. It does not make additional REST calls or change row-level routing status.

Microsoft documentation: [Delta Lake schema enforcement](https://learn.microsoft.com/fabric/data-engineering/delta-lake-schema-evolution#what-schema-enforcement-means) explains why the DataFrame schema must match the target table unless schema changes are explicitly enabled.


In [ ]:
# Convert CSV strings and enrichment results to an explicit Spark schema.
# Missing, invalid, or non-finite numeric values become null for Delta storage.
numeric_columns = {
    "Plant_Latitude", "Plant_Longitude", "ManfactSupplier_Latitude",
    "ManfactSupplier_Longitude", "Part_Qty", "Part_NetWeight_kg", "Distance_in_km",
    "Emissions_factor_kgCO2_per_km", "Trip_multiplier", "Driving_distance_km",
    "CO2_emitted_kg", "Carbon_cost_USD", "Carbon_price_USD_per_tonne",
    "Supplier_route_latitude", "Supplier_route_longitude",
    "Plant_route_latitude", "Plant_route_longitude",
}
columns = sorted({key for row in enriched for key in row})
schema = StructType([
    StructField(name, DoubleType() if name in numeric_columns else StringType(), True)
    for name in columns
])
typed = []
for row in enriched:
    converted: dict[str, Any] = {}
    for name in columns:
        value = row.get(name)
        if value in ("", None):
            converted[name] = None
        elif name in numeric_columns:
            try:
                numeric = float(value)
                converted[name] = numeric if math.isfinite(numeric) else None
            except (ValueError, TypeError):
                converted[name] = None
        else:
            converted[name] = str(value)
    typed.append(converted)
print_stage("typed", typed)


## Step 6: Write delivery and route-point tables

`route_points(...)` expands each successful road geometry into ordered vertices. `Path_ID` identifies a delivery segment, and `Point_sequence` preserves vertex order for the Power BI map. Failed routes have no vertices.

> [!WARNING]
> The next cell overwrites the attached Lakehouse's `deliveries` and `route_points` Delta tables, including their schemas. It replaces the current snapshot rather than appending history.

`deliveries` includes both successful and failed rows for review. `route_points` contains only successful route geometry. These writes are separate operations, not a single transaction across both tables.

Microsoft documentation: [Replace a Delta table schema with `overwriteSchema`](https://learn.microsoft.com/fabric/data-engineering/delta-lake-schema-evolution#use-overwrite-schema-to-replace-the-table-schema), including the impact on downstream consumers.


In [ ]:
points = route_points(enriched)
point_schema = StructType([
    StructField("Delivery_ID", StringType(), False),
    StructField("Path_ID", StringType(), False),
    StructField("Route_segment", LongType(), False),
    StructField("Point_sequence", LongType(), False),
    StructField("Longitude", DoubleType(), False),
    StructField("Latitude", DoubleType(), False),
])
# Replace the two Lakehouse Delta tables for this refresh, including
# ERROR rows in deliveries but only successful road vertices in route_points.
df = spark.createDataFrame(typed, schema)
route_df = spark.createDataFrame(points, point_schema)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("deliveries")
route_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("route_points")
print_stage("saved_deliveries", typed)
print_stage("saved_route_points", points)


## Step 7: Export routes and check the refresh summary

Write successful routes to `Files/sample/routes.geojson` as a GeoJSON `FeatureCollection`, with delivery ID and carbon values on each feature. Write aggregate results to `Files/sample/refresh_summary.json` and print the same summary. Both exports replace their previous contents.

Illustrative summary for one successful delivery with two exported route vertices:

```json
{
  "deliveries": 1,
  "successful": 1,
  "route_points": 2,
  "road_km": 12.3456,
  "co2_kg": 6.666624,
  "carbon_cost_usd": 0.6666624,
  "carbon_price_usd_per_tonne": 100.0
}
```

These files are notebook outputs, not REST responses. Totals treat missing metrics as zero. Compare `successful` with `deliveries` and inspect `Route_Error_Reason` and `Route_Error_Detail` in the delivery table for failed rows.

If no deliveries succeed, the cell raises an error after persisting the tables and exports. A partially successful refresh does not raise this final error, so check the counts before using the results.

Microsoft documentation: [Write a UTF-8 file with `notebookutils.fs.put`](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-file-system#write-file), including the overwrite flag used for both exports.


In [ ]:
# Export successful routes as GeoJSON plus a JSON refresh summary.
# These are local enrichment outputs, not additional Azure Maps REST calls.
features = [
    {
        "type": "Feature",
        "geometry": json.loads(row["Route_geojson"]),
        "properties": {
            "Delivery_ID": row["Delivery_ID"],
            "CO2_emitted_kg": row["CO2_emitted_kg"],
            "Carbon_cost_USD": row["Carbon_cost_USD"],
        },
    }
    for row in enriched if row["Route_Status"] == "OK"
]
notebookutils.fs.put(
    "Files/sample/routes.geojson",
    json.dumps({"type": "FeatureCollection", "features": features}, allow_nan=False),
    True,
)
summary = {
    "deliveries": len(enriched),
    "successful": sum(row["Route_Status"] == "OK" for row in enriched),
    "route_points": len(points),
    "road_km": sum(row["Driving_distance_km"] or 0 for row in enriched),
    "co2_kg": sum(row["CO2_emitted_kg"] or 0 for row in enriched),
    "carbon_cost_usd": sum(row["Carbon_cost_USD"] or 0 for row in enriched),
    "carbon_price_usd_per_tonne": CARBON_PRICE_USD_PER_TONNE,
}
notebookutils.fs.put("Files/sample/refresh_summary.json", json.dumps(summary), True)
print_stage("exported_routes", features)
print(json.dumps(summary, indent=2))
# Persist diagnostics first, then fail the refresh if every delivery failed.
if not summary["successful"]:
    raise RuntimeError("No routes succeeded. Error rows persisted; check Maps identity and permissions.")
